# Member 3: PubMed experiments & result aggregation

**Project:** Do Graph Neural Networks Really Beat Counting Common Neighbours? (Milestone 1)

This notebook:
1. Runs **GAE**, **VGAE** and the **Adamic–Adar** + **Common Neighbors** baselines on **PubMed**, 10 random splits each.
   For each seed, all models are evaluated on the **same** hidden test edges, so the comparison is fair.
2. Merges these results with the Cora / CiteSeer results from Members 1–2.
3. Computes **mean ± std** for every model–dataset pair and compares with Table 1 of Kipf & Welling (2016).

Model code mirrors PyG's official `examples/autoencoder.py` (same encoder, 16-dim embedding, lr 0.01, 400 epochs,
`RandomLinkSplit` with 5% val / 10% test, feature normalisation).

**Runtime:** `Runtime → Change runtime type → T4 GPU` is recommended (CPU also works, just slower).

In [1]:
!pip install -q torch_geometric
import torch, torch_geometric
print('torch', torch.__version__, '| PyG', torch_geometric.__version__, '| CUDA:', torch.cuda.is_available())

torch 2.11.0+cu130 | PyG 2.8.1 | CUDA: True


In [2]:
import random, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GAE, VGAE, GCNConv
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- Settings (same as PyG examples/autoencoder.py) ----
# Datasets to run HERE. 'PubMed' is Member 3's part. Add 'Cora', 'CiteSeer' to re-run them with the
# exact same settings (~2 extra minutes on a T4); their results then replace Member 2's in the table.
DATASETS   = ['Cora', 'CiteSeer', 'PubMed']
N_RUNS     = 10
SEEDS      = list(range(N_RUNS))   # seeds 0..9 -> 10 different random edge splits
EPOCHS     = 400
LR         = 0.01
OUT_CH     = 16

## Models (identical to the official example)

In [3]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv2 = GCNConv(2 * out_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)


class VariationalGCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv_mu = GCNConv(2 * out_channels, out_channels)
        self.conv_logstd = GCNConv(2 * out_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

## Data split, training, and the heuristic baseline

* `make_split(seed)` creates one random train/val/test edge split. Test (and val) edges are **removed** from
  `train_data.edge_index`, so neither the GNN nor the heuristic ever sees them.
* `adamic_adar` / `common_neighbours` are computed **only on the training graph**, then scored on the exact same
  test positive and negative pairs the GNN is tested on.

In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_split(dataset_name, seed):
    set_seed(seed)
    transform = T.Compose([
        T.NormalizeFeatures(),
        T.ToDevice(device),
        T.RandomLinkSplit(num_val=0.05, num_test=0.1, is_undirected=True,
                          split_labels=True, add_negative_train_samples=False),
    ])
    dataset = Planetoid('/content/data/Planetoid', dataset_name, transform=transform)
    return dataset[0], dataset.num_features   # (train_data, val_data, test_data)


def train_gnn(variational, train_data, test_data, num_features, seed):
    set_seed(seed)
    enc = VariationalGCNEncoder if variational else GCNEncoder
    model = (VGAE if variational else GAE)(enc(num_features, OUT_CH)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train(); opt.zero_grad()
        z = model.encode(train_data.x, train_data.edge_index)
        loss = model.recon_loss(z, train_data.pos_edge_label_index)
        if variational:
            loss = loss + (1 / train_data.num_nodes) * model.kl_loss()
        loss.backward(); opt.step()

    # final-epoch test score (what the example prints at the last epoch)
    model.eval()
    with torch.no_grad():
        z = model.encode(test_data.x, test_data.edge_index)
        auc, ap = model.test(z, test_data.pos_edge_label_index, test_data.neg_edge_label_index)
    return auc, ap


def heuristic_scores(train_data, test_data):
    """Common Neighbors and Adamic-Adar on the TRAINING graph only."""
    n = train_data.num_nodes
    ei = train_data.edge_index.cpu().numpy()
    A = sp.csr_matrix((np.ones(ei.shape[1]), (ei[0], ei[1])), shape=(n, n))
    A.data[:] = 1.0                       # binarise (guards against duplicates)
    deg = np.asarray(A.sum(1)).ravel()
    w = np.zeros(n)
    mask = deg > 1
    w[mask] = 1.0 / np.log(deg[mask])     # AA weight 1/log(deg); deg<=1 nodes can't be common neighbours of 2 nodes
    A_w = A @ sp.diags(w)

    pos = test_data.pos_edge_label_index.cpu().numpy()
    neg = test_data.neg_edge_label_index.cpu().numpy()
    pairs = np.concatenate([pos, neg], axis=1)
    y = np.concatenate([np.ones(pos.shape[1]), np.zeros(neg.shape[1])])
    u, v = pairs

    cn = np.asarray(A[u].multiply(A[v]).sum(1)).ravel()
    aa = np.asarray(A_w[u].multiply(A[v]).sum(1)).ravel()
    out = {}
    for name, s in [('Common Neighbors', cn), ('Adamic-Adar', aa)]:
        out[name] = (roc_auc_score(y, s), average_precision_score(y, s))
    return out

## Sanity check: test edges are hidden from training

In [5]:
train_data, val_data, test_data = make_split(DATASETS[0], 0)[0]
train_set = set(map(tuple, train_data.edge_index.t().tolist()))
leak = sum((u, v) in train_set or (v, u) in train_set
           for u, v in test_data.pos_edge_label_index.t().tolist())
print('Train message-passing edges:', train_data.edge_index.size(1))
print('Test positive edges:', test_data.pos_edge_label_index.size(1),
      '| test negatives:', test_data.neg_edge_label_index.size(1))
print('Test edges leaking into training graph:', leak)
assert leak == 0

Processing...
Done!


Train message-passing edges: 8976
Test positive edges: 527 | test negatives: 527
Test edges leaking into training graph: 0


## Run all 10 splits per dataset

Each seed gives one split; GAE, VGAE, Adamic–Adar and Common Neighbors are all evaluated on it.
Raw per-run results are saved to `member3_raw_results.csv`.

In [6]:
rows = []
t0 = time.time()
for ds in DATASETS:
    for run, seed in enumerate(SEEDS, start=1):
        (train_data, val_data, test_data), nfeat = make_split(ds, seed)

        for name, (auc, ap) in heuristic_scores(train_data, test_data).items():
            rows.append(dict(dataset=ds, model=name, run=run, seed=seed, auc=auc, ap=ap))
        for name, variational in [('GAE', False), ('VGAE', True)]:
            auc, ap = train_gnn(variational, train_data, test_data, nfeat, seed)
            rows.append(dict(dataset=ds, model=name, run=run, seed=seed, auc=auc, ap=ap))

        r = [x for x in rows if x['run'] == run and x['dataset'] == ds]
        print(f'{ds} run {run:2d} | ' +
              ' | '.join(f"{x['model']}: AUC {100*x['auc']:.2f} AP {100*x['ap']:.2f}" for x in r) +
              f' | {time.time()-t0:.0f}s')

raw_df = pd.DataFrame(rows)
raw_df.to_csv('member3_raw_results.csv', index=False)
raw_df.head()

Cora run  1 | Common Neighbors: AUC 72.88 AP 72.66 | Adamic-Adar: AUC 72.87 AP 72.74 | GAE: AUC 89.92 AP 90.41 | VGAE: AUC 88.50 AP 89.07 | 15s
Cora run  2 | Common Neighbors: AUC 71.31 AP 71.13 | Adamic-Adar: AUC 71.32 AP 71.18 | GAE: AUC 89.81 AP 90.11 | VGAE: AUC 89.74 AP 89.36 | 23s
Cora run  3 | Common Neighbors: AUC 71.90 AP 71.69 | Adamic-Adar: AUC 71.93 AP 71.88 | GAE: AUC 90.98 AP 91.70 | VGAE: AUC 89.33 AP 89.43 | 31s
Cora run  4 | Common Neighbors: AUC 72.66 AP 72.16 | Adamic-Adar: AUC 72.86 AP 73.03 | GAE: AUC 92.65 AP 93.34 | VGAE: AUC 91.25 AP 91.57 | 39s
Cora run  5 | Common Neighbors: AUC 70.23 AP 69.76 | Adamic-Adar: AUC 70.34 AP 70.41 | GAE: AUC 89.68 AP 89.74 | VGAE: AUC 87.59 AP 88.17 | 47s
Cora run  6 | Common Neighbors: AUC 73.24 AP 73.05 | Adamic-Adar: AUC 73.35 AP 73.56 | GAE: AUC 90.95 AP 91.17 | VGAE: AUC 88.66 AP 88.82 | 55s
Cora run  7 | Common Neighbors: AUC 71.45 AP 71.41 | Adamic-Adar: AUC 71.48 AP 71.53 | GAE: AUC 91.34 AP 92.16 | VGAE: AUC 89.79 AP 90.3

Processing...
Done!


CiteSeer run  1 | Common Neighbors: AUC 66.92 AP 66.92 | Adamic-Adar: AUC 66.92 AP 66.92 | GAE: AUC 90.16 AP 90.72 | VGAE: AUC 86.69 AP 87.50 | 102s
CiteSeer run  2 | Common Neighbors: AUC 65.60 AP 65.60 | Adamic-Adar: AUC 65.60 AP 65.60 | GAE: AUC 90.04 AP 91.06 | VGAE: AUC 88.84 AP 89.31 | 110s
CiteSeer run  3 | Common Neighbors: AUC 67.25 AP 67.25 | Adamic-Adar: AUC 67.25 AP 67.25 | GAE: AUC 90.68 AP 92.02 | VGAE: AUC 89.28 AP 90.61 | 118s
CiteSeer run  4 | Common Neighbors: AUC 67.47 AP 67.47 | Adamic-Adar: AUC 67.47 AP 67.47 | GAE: AUC 88.68 AP 90.08 | VGAE: AUC 86.54 AP 88.21 | 126s
CiteSeer run  5 | Common Neighbors: AUC 66.28 AP 66.23 | Adamic-Adar: AUC 66.29 AP 66.34 | GAE: AUC 88.67 AP 89.32 | VGAE: AUC 86.20 AP 86.41 | 133s
CiteSeer run  6 | Common Neighbors: AUC 67.47 AP 67.47 | Adamic-Adar: AUC 67.47 AP 67.47 | GAE: AUC 88.29 AP 88.79 | VGAE: AUC 85.88 AP 85.92 | 141s
CiteSeer run  7 | Common Neighbors: AUC 64.76 AP 64.66 | Adamic-Adar: AUC 64.82 AP 65.04 | GAE: AUC 88.15 

Processing...
Done!


PubMed run  1 | Common Neighbors: AUC 65.18 AP 65.15 | Adamic-Adar: AUC 65.19 AP 65.20 | GAE: AUC 96.83 AP 96.97 | VGAE: AUC 94.87 AP 95.07 | 210s
PubMed run  2 | Common Neighbors: AUC 64.51 AP 64.50 | Adamic-Adar: AUC 64.52 AP 64.53 | GAE: AUC 96.63 AP 96.89 | VGAE: AUC 94.78 AP 95.06 | 240s
PubMed run  3 | Common Neighbors: AUC 64.28 AP 64.26 | Adamic-Adar: AUC 64.29 AP 64.32 | GAE: AUC 96.25 AP 96.39 | VGAE: AUC 95.59 AP 95.70 | 269s
PubMed run  4 | Common Neighbors: AUC 64.20 AP 64.18 | Adamic-Adar: AUC 64.21 AP 64.23 | GAE: AUC 96.56 AP 96.78 | VGAE: AUC 94.89 AP 95.21 | 299s
PubMed run  5 | Common Neighbors: AUC 64.40 AP 64.37 | Adamic-Adar: AUC 64.41 AP 64.41 | GAE: AUC 96.53 AP 96.63 | VGAE: AUC 94.60 AP 94.72 | 328s
PubMed run  6 | Common Neighbors: AUC 64.33 AP 64.24 | Adamic-Adar: AUC 64.34 AP 64.34 | GAE: AUC 96.77 AP 96.61 | VGAE: AUC 94.54 AP 94.58 | 357s
PubMed run  7 | Common Neighbors: AUC 63.86 AP 63.83 | Adamic-Adar: AUC 63.85 AP 63.80 | GAE: AUC 96.61 AP 96.79 | VGA

,dataset,model,run,seed,auc,ap
0,Cora,Common Neighbors,1,0,0.728763,0.726624
1,Cora,Adamic-Adar,1,0,0.728723,0.727381
2,Cora,GAE,1,0,0.899193,0.904122
3,Cora,VGAE,1,0,0.885035,0.890738
4,Cora,Common Neighbors,2,1,0.713064,0.711313


## Cora / CiteSeer results from Members 1–2

They handed over **summary statistics only** (mean ± std over 10 runs, in %), so they are entered as-is.
Source: `Experiment_Report.docx` (Tables 1–2) and `member2_citeseer_handover.txt`.

**Note:** the handover lists **200 training epochs**, while the official example and this notebook use **400**.
Any dataset listed in `DATASETS` is taken from this notebook's own runs instead of this cell.

In [7]:
# (dataset, model): (AUC mean, AUC std, AP mean, AP std, n_runs)  - all in %
MEMBER2 = {
    ('Cora', 'GAE'):                  (89.84, 1.20, 90.14, 1.29, 10),
    ('Cora', 'VGAE'):                 (83.27, 2.43, 83.58, 2.33, 10),
    ('Cora', 'Common Neighbors'):     (71.33, 0.64, 71.06, 0.62, 10),
    ('Cora', 'Adamic-Adar'):          (71.40, 0.63, 71.45, 0.58, 10),
    ('CiteSeer', 'GAE'):              (87.80, 1.37, 88.26, 1.30, 10),
    ('CiteSeer', 'VGAE'):             (79.43, 3.32, 80.32, 2.90, 10),
    ('CiteSeer', 'Common Neighbors'): (67.15, 0.76, 67.10, 0.75, 10),
    ('CiteSeer', 'Adamic-Adar'):      (67.16, 0.76, 67.19, 0.72, 10),
}
m2 = pd.DataFrame([dict(dataset=d, model=m, n=n, AUC_mean=a, AUC_std=sa, AP_mean=p, AP_std=sp_,
                        source='Members 1-2')
                   for (d, m), (a, sa, p, sp_, n) in MEMBER2.items()])
m2

,dataset,model,n,AUC_mean,AUC_std,AP_mean,AP_std,source
0,Cora,GAE,10,89.84,1.20,90.14,1.29,Members 1-2
1,Cora,VGAE,10,83.27,2.43,83.58,2.33,Members 1-2
2,Cora,Common Neighbors,10,71.33,0.64,71.06,0.62,Members 1-2
3,Cora,Adamic-Adar,10,71.40,0.63,71.45,0.58,Members 1-2
4,CiteSeer,GAE,10,87.80,1.37,88.26,1.30,Members 1-2
5,CiteSeer,VGAE,10,79.43,3.32,80.32,2.90,Members 1-2
6,CiteSeer,Common Neighbors,10,67.15,0.76,67.10,0.75,Members 1-2
7,CiteSeer,Adamic-Adar,10,67.16,0.76,67.19,0.72,Members 1-2


## Aggregate: mean ± std and comparison with the paper

For runs in this notebook, std is the sample standard deviation over the 10 runs (`ddof=1`).
Paper targets: Table 1 of Kipf & Welling, *Variational Graph Auto-Encoders* (2016).
A result counts as reproduced if both AUC and AP are within max(1 point, 1 std) of the paper.

In [8]:
PAPER = {  # (dataset, model): (AUC, AP)
    ('Cora', 'GAE'): (91.0, 92.0),     ('Cora', 'VGAE'): (91.4, 92.6),
    ('CiteSeer', 'GAE'): (89.5, 89.9), ('CiteSeer', 'VGAE'): (90.8, 92.0),
    ('PubMed', 'GAE'): (96.4, 96.5),   ('PubMed', 'VGAE'): (94.4, 94.7),
}

g = raw_df.groupby(['dataset', 'model'])
own = pd.DataFrame({
    'n':        g.size(),
    'AUC_mean': g['auc'].mean() * 100, 'AUC_std': g['auc'].std(ddof=1) * 100,
    'AP_mean':  g['ap'].mean() * 100,  'AP_std':  g['ap'].std(ddof=1) * 100,
}).reset_index()
own['source'] = 'Member 3 (this notebook)'

summary = pd.concat([m2[~m2.dataset.isin(DATASETS)], own], ignore_index=True)

ds_order = ['Cora', 'CiteSeer', 'PubMed']
m_order = ['Common Neighbors', 'Adamic-Adar', 'GAE', 'VGAE']
summary['dataset'] = pd.Categorical(summary['dataset'], ds_order, ordered=True)
summary['model'] = pd.Categorical(summary['model'], m_order, ordered=True)
summary = summary.sort_values(['dataset', 'model']).reset_index(drop=True)

summary['Paper AUC'] = [PAPER.get((d, m), (np.nan, np.nan))[0] for d, m in zip(summary.dataset, summary.model)]
summary['Paper AP']  = [PAPER.get((d, m), (np.nan, np.nan))[1] for d, m in zip(summary.dataset, summary.model)]
summary['ΔAUC'] = summary['AUC_mean'] - summary['Paper AUC']
summary['ΔAP']  = summary['AP_mean'] - summary['Paper AP']
summary['Reproduced?'] = np.where(summary['Paper AUC'].isna(), '—',
    np.where((summary['ΔAUC'].abs() <= np.maximum(1.0, summary['AUC_std'])) &
             (summary['ΔAP'].abs()  <= np.maximum(1.0, summary['AP_std'])), 'yes', 'no'))
summary.to_csv('summary_results.csv', index=False)

fmt = lambda m, s: f'{m:.1f} ± {s:.1f}'
table = pd.DataFrame({
    'Dataset': summary['dataset'].astype(str),
    'Model': summary['model'].astype(str),
    'AUC (%)': [fmt(m, s) for m, s in zip(summary.AUC_mean, summary.AUC_std)],
    'AP (%)':  [fmt(m, s) for m, s in zip(summary.AP_mean, summary.AP_std)],
    'Paper AUC / AP': [('—' if np.isnan(a) else f'{a:.1f} / {p:.1f}')
                       for a, p in zip(summary['Paper AUC'], summary['Paper AP'])],
    'Reproduced?': summary['Reproduced?'],
    'Source': summary['source'],
})
table

,Dataset,Model,AUC (%),AP (%),Paper AUC / AP,Reproduced?,Source
0,Cora,Common Neighbors,71.9 ± 0.9,71.7 ± 1.0,—,—,Member 3 (this notebook)
1,Cora,Adamic-Adar,72.0 ± 0.9,72.0 ± 1.0,—,—,Member 3 (this notebook)
2,Cora,GAE,90.7 ± 0.9,91.3 ± 1.1,91.0 / 92.0,yes,Member 3 (this notebook)
3,Cora,VGAE,89.1 ± 1.1,89.5 ± 0.9,91.4 / 92.6,no,Member 3 (this notebook)
4,CiteSeer,Common Neighbors,66.5 ± 0.9,66.4 ± 1.0,—,—,Member 3 (this notebook)
5,CiteSeer,Adamic-Adar,66.5 ± 0.9,66.5 ± 0.9,—,—,Member 3 (this notebook)
6,CiteSeer,GAE,88.8 ± 1.1,89.6 ± 1.3,89.5 / 89.9,yes,Member 3 (this notebook)
7,CiteSeer,VGAE,86.6 ± 1.7,87.3 ± 1.9,90.8 / 92.0,no,Member 3 (this notebook)
8,PubMed,Common Neighbors,64.3 ± 0.4,64.3 ± 0.4,—,—,Member 3 (this notebook)
9,PubMed,Adamic-Adar,64.3 ± 0.4,64.3 ± 0.4,—,—,Member 3 (this notebook)


## Export the table (Markdown for the report, plus LaTeX)

In [9]:
print(table.to_markdown(index=False))
print()
print(table.to_latex(index=False, escape=True))

| Dataset   | Model            | AUC (%)    | AP (%)     | Paper AUC / AP   | Reproduced?   | Source                   |
|:----------|:-----------------|:-----------|:-----------|:-----------------|:--------------|:-------------------------|
| Cora      | Common Neighbors | 71.9 ± 0.9 | 71.7 ± 1.0 | —                | —             | Member 3 (this notebook) |
| Cora      | Adamic-Adar      | 72.0 ± 0.9 | 72.0 ± 1.0 | —                | —             | Member 3 (this notebook) |
| Cora      | GAE              | 90.7 ± 0.9 | 91.3 ± 1.1 | 91.0 / 92.0      | yes           | Member 3 (this notebook) |
| Cora      | VGAE             | 89.1 ± 1.1 | 89.5 ± 0.9 | 91.4 / 92.6      | no            | Member 3 (this notebook) |
| CiteSeer  | Common Neighbors | 66.5 ± 0.9 | 66.4 ± 1.0 | —                | —             | Member 3 (this notebook) |
| CiteSeer  | Adamic-Adar      | 66.5 ± 0.9 | 66.5 ± 0.9 | —                | —             | Member 3 (this notebook) |
| CiteSeer  | GAE              |

In [10]:
# GNN vs best heuristic, per dataset
best = summary.pivot(index='dataset', columns='model', values='AUC_mean')
heur = [c for c in ['Common Neighbors', 'Adamic-Adar'] if c in best.columns]
for d in best.index:
    row = best.loc[d]
    h = max(heur, key=lambda c: row[c] if not np.isnan(row[c]) else -1)
    msg = ', '.join(f'{m} {row[m] - row[h]:+.1f}' for m in ['GAE', 'VGAE'] if m in row and not np.isnan(row[m]))
    print(f'{d}: best heuristic = {h} ({row[h]:.1f} AUC); GNN margin over it: {msg}')

Cora: best heuristic = Adamic-Adar (72.0 AUC); GNN margin over it: GAE +18.7, VGAE +17.1
CiteSeer: best heuristic = Adamic-Adar (66.5 AUC); GNN margin over it: GAE +22.3, VGAE +20.1
PubMed: best heuristic = Adamic-Adar (64.3 AUC); GNN margin over it: GAE +32.3, VGAE +30.8


In [11]:
# Download all result files
try:
    from google.colab import files
    for f in ['member3_raw_results.csv', 'summary_results.csv']:
        files.download(f)
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>